# Блок 7. Основы искусственного интеллекта и машинного обучения
# Занятие 5. Классификация и Logistic Regression

## Что такое классификация

Если регрессия предсказывает число, то классификация предсказывает категорию.

Примеры:

- купит клиент или нет;
- спам или не спам;
- болен или здоров;
- сдаст студент экзамен или нет.

## Бинарная классификация

Самый простой случай:

- 0 = Нет
- 1 = Да

## Logistic Regression

Несмотря на название, это алгоритм классификации.

Модель вычисляет вероятность принадлежности объекта к классу.

## Датасет занятия

Используем датасет Breast Cancer из Scikit-Learn.

Задача модели:

- 0 = злокачественное образование
- 1 = доброкачественное образование

## Связь с дипломом

Классификация используется в:

- HR;
- медицине;
- финансах;
- продажах;
- маркетинге.


Сначала изучите SOLVED-ноутбук вместе с преподавателем, затем выполните TODO самостоятельно.

## Ячейка 1. Загрузка датасета

### Алгоритм
Загружаем встроенный датасет классификации.

In [97]:
import pandas as pd
from IPython.display import display

url = "../../data/daily_meal_table.csv"
df: pd.DataFrame = pd.read_csv(url)


#'energy_kcal', "products"
feature_cols = [ 'protein_g', 'fat_g', 'carbohydrates_g', 'fiber_g', 'sugars_g', 'saturated_fat_g',
                'cholesterol_mg', 'sodium_mg', 'potassium_mg', 'water_g', ]
target_col = "is_main_meal"

df[target_col] = (df["meal"] != "breakfast").astype("int8")

print(df.shape)
display(df.head())

assert target_col in df.columns
assert len(df) > 1000
assert df.shape[1] == 16

(15322, 16)


,id,date,meal,products,energy_kcal,protein_g,fat_g,carbohydrates_g,fiber_g,sugars_g,saturated_fat_g,cholesterol_mg,sodium_mg,potassium_mg,water_g,is_main_meal
0,0060b6899180e194092d60a0e50aeea1,2019/2/22,breakfast,2,615.0,16.39,17.35,102.97,9.4,28.38,2.796,16.0,954.0,165.0,50.60,0
1,0060b6899180e194092d60a0e50aeea1,2019/2/22,lunch,3,564.0,33.00,10.00,85.40,8.2,0.00,0.555,0.0,248.0,739.0,164.80,1
2,0060b6899180e194092d60a0e50aeea1,2019/2/22,supper,2,239.0,15.73,10.53,22.27,1.9,3.85,1.780,28.0,536.0,128.0,84.00,1
3,0060b6899180e194092d60a0e50aeea1,2019/2/23,breakfast,3,968.0,17.68,50.74,110.79,8.2,2.00,8.091,20.0,1600.0,563.0,115.20,0
4,0060b6899180e194092d60a0e50aeea1,2019/2/23,lunch,3,487.0,16.27,6.06,92.61,7.3,2.79,1.040,12.0,512.0,614.0,182.47,1


## Ячейка 2. Изучение данных

### Алгоритм
Смотрим признаки и классы.

In [98]:
X: pd.DataFrame = pd.DataFrame(df, columns=feature_cols)
y: pd.Series = pd.Series(df[target_col])


assert len(X) == len(y)

## Ячейка 3. Распределение классов

### Алгоритм
Считаем объекты каждого класса.

In [99]:
class_counts = y.value_counts()

print(class_counts)

assert class_counts.sum() == len(y)

is_main_meal
1    10165
0     5157
Name: count, dtype: int64


## Ячейка 4. Train/Test Split

### Алгоритм
Разделяем данные.

In [100]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df["id"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(len(X_train) / len(X_test))
assert abs(len(X_train) / len(X_test) - 4) < 1

4.169365721997301


## Ячейка 5. Масштабирование

### Алгоритм
Подготавливаем признаки.

In [101]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

assert X_train_scaled.shape[0] == len(X_train)

## Ячейка 6. Обучение Logistic Regression

### Алгоритм
Строим модель.

In [102]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=5000, class_weight="balanced")

model.fit(X_train_scaled, y_train)


coefs = pd.DataFrame({
    "feature": feature_cols,
    "coef": model.coef_[0],
})
coefs["abs_coef"] = coefs["coef"].abs()
coefs = coefs.sort_values("abs_coef", ascending=False).reset_index(drop=True)

print(coefs.to_string(index=False))


assert model is not None

        feature      coef  abs_coef
saturated_fat_g -1.360396  1.360396
carbohydrates_g -0.948799  0.948799
   potassium_mg  0.840161  0.840161
      protein_g  0.693906  0.693906
      sodium_mg  0.563143  0.563143
        water_g  0.392892  0.392892
 cholesterol_mg -0.179074  0.179074
        fiber_g -0.033407  0.033407
       sugars_g  0.024174  0.024174
          fat_g -0.014165  0.014165


## Ячейка 7. Предсказание классов

### Алгоритм
Получаем прогноз.

In [103]:
predictions = model.predict(X_test_scaled)
print(predictions[:10])

assert len(predictions) == len(y_test)

[0 1 1 1 0 1 0 1 1 0]


## Ячейка 8. Вероятности

### Алгоритм
Получаем вероятность классов.

In [104]:
probabilities = model.predict_proba(X_test_scaled)

print(probabilities[:5])

assert probabilities.shape[1] == 2

[[0.68714099 0.31285901]
 [0.15577413 0.84422587]
 [0.42747215 0.57252785]
 [0.40507367 0.59492633]
 [0.50291153 0.49708847]]


## Ячейка 9. Метрики классификации

### Алгоритм
Считаем Accuracy, Precision, Recall, F1.

In [105]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("Accuracy", accuracy)
print("Precision", precision)
print("Recall", recall)
print("F1", f1)

assert accuracy > 0.8

Accuracy 0.8039811066126855
Precision 0.8798024149286499
Recall 0.8157760814249364
F1 0.8465804066543438


## Ячейка 10. Мини-проект

### Алгоритм
Формируем итоговый отчет.

In [107]:
report = {
    "rows": len(X),
    "features": X.shape[1],
    "accuracy": round(accuracy,3),
    "precision": round(precision,3),
    "recall": round(recall,3),
    "f1": round(f1,3)
}

print(report)

assert report["features"] > 1


{'rows': 15322, 'features': 10, 'accuracy': 0.804, 'precision': 0.88, 'recall': 0.816, 'f1': 0.847}
